# QUESTION 1


## Q.1. Write cudaC code (for block size of 8 threads) as follows:-  (CLO-1, C2) 
• Given input vectors hA & hB each of N = 60 elements of integer data type 
• Initialize vectors hA & hB with random values between low=10 & high=99 
• For host names hA, hB, hC, hk, hProd, hSum/ hMax/ hMin, hAvg & hB₁, hB₂, 
hB₃, hB₄, use corresponding device names dA, dB, dC, dk, dProd, dSum/ 
dMax/ dMin, dAvg, dB₁, dB₂, dB₃, dB₄ 
Q1: Add/Subtract/element-wise multiply of two vectors hA & hB, put results in hC 

In [1]:
%%writefile q1.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N    60
#define LOW  10
#define HIGH 99
#define BLOCK_SIZE 8

// ── CUDA Kernels
__global__ void vecAddKernel(int *dA, int *dB, int *dC, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        dC[i] = dA[i] + dB[i];
}

__global__ void vecSubKernel(int *dA, int *dB, int *dC, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        dC[i] = dA[i] - dB[i];
}

__global__ void vecMulKernel(int *dA, int *dB, int *dC, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        dC[i] = dA[i] * dB[i];
}

// ── Host Functions 
__host__ void vecAdd(int *hA, int *hB, int *hC, int n)
{
    int *dA, *dB, *dC;
    int size = n * sizeof(int);

    cudaMalloc((void**)&dA, size);
    cudaMalloc((void**)&dB, size);
    cudaMalloc((void**)&dC, size);

    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, size, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    vecAddKernel<<<DimGrid, DimBlock>>>(dA, dB, dC, n);

    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
}

__host__ void vecSub(int *hA, int *hB, int *hC, int n)
{
    int *dA, *dB, *dC;
    int size = n * sizeof(int);

    cudaMalloc((void**)&dA, size);
    cudaMalloc((void**)&dB, size);
    cudaMalloc((void**)&dC, size);

    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, size, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    vecSubKernel<<<DimGrid, DimBlock>>>(dA, dB, dC, n);

    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
}

__host__ void vecMul(int *hA, int *hB, int *hC, int n)
{
    int *dA, *dB, *dC;
    int size = n * sizeof(int);

    cudaMalloc((void**)&dA, size);
    cudaMalloc((void**)&dB, size);
    cudaMalloc((void**)&dC, size);

    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, size, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    vecMulKernel<<<DimGrid, DimBlock>>>(dA, dB, dC, n);

    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
}

// ── Main 
int main()
{
    int *hA, *hB, *hC;

    hA = (int*) malloc(N * sizeof(int));
    hB = (int*) malloc(N * sizeof(int));
    hC = (int*) malloc(N * sizeof(int));

    srand(time(NULL));
    for(int i=0; i<N; i++)
    {
        hA[i] = rand()%90+10;
        hB[i] = rand()%90+10;
    }

    printf("hA: "); for(int i=0;i<N;i++) printf("%4d",hA[i]); printf("\n");
    printf("hB: "); for(int i=0;i<N;i++) printf("%4d",hB[i]); printf("\n");

    // Addition
    vecAdd(hA, hB, hC, N);
    printf("\nhC = hA + hB:\n");
    for(int i=0;i<N;i++) printf("%4d",hC[i]); printf("\n");

    // Subtraction
    vecSub(hA, hB, hC, N);
    printf("\nhC = hA - hB:\n");
    for(int i=0;i<N;i++) printf("%4d",hC[i]); printf("\n");

    // Element-wise Multiply
    vecMul(hA, hB, hC, N);
    printf("\nhC = hA * hB:\n");
    for(int i=0;i<N;i++) printf("%4d",hC[i]); printf("\n");

    free(hA); free(hB); free(hC);
    return 0;
}

Writing q1.cu


In [2]:
!nvcc -arch=sm_75 q1.cu -o q1

!./q1

hA:   70  30  50  77  58  65  32  35  44  79  31  69  86  69  96  55  74  28  46  41  35  86  67  12  25  12  26  99  48  79  23  62  77  64  90  18  16  86  68  89  45  45  25  40  26  21  96  79  68  49  52  35  51  85  75  18  28  34  49  20
hB:   42  99  56  65  77  92  18  93  51  86  82  10  24  63  41  66  38  76  57  94  59  19  22  63  43  47  33  64  30  44  87  87  42  23  57  77  94  74  50  46  91  23  44  25  19  39  35  73  95  58  57  59  73  72  74  20  53  95  26  27

hC = hA + hB:
 112 129 106 142 135 157  50 128  95 165 113  79 110 132 137 121 112 104 103 135  94 105  89  75  68  59  59 163  78 123 110 149 119  87 147  95 110 160 118 135 136  68  69  65  45  60 131 152 163 107 109  94 124 157 149  38  81 129  75  47

hC = hA - hB:
  28 -69  -6  12 -19 -27  14 -58  -7  -7 -51  59  62   6  55 -11  36 -48 -11 -53 -24  67  45 -51 -18 -35  -7  35  18  35 -64 -25  35  41  33 -59 -78  12  18  43 -46  22 -19  15   7 -18  61   6 -27  -9  -5 -24 -22  13   1  -2 -25 -61  23  -